<a href="https://colab.research.google.com/github/Aisyaaswy/Pemrograman_Mesin/blob/main/jobsheet_6/dataset_pbl/HNSW/praktikum_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **PRAKTIKUM 4**
Percobaan kali ini kita akan melihat perbedaan ketiga model yang telah kita bahas dan bandingkan hasilnya.


In [1]:
!pip install annoy faiss-cpu hnswlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 7.1 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 63.1 MB/s  0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520035 sha256=2a921cdda1e9f852e3e6e5da899be74691d15acc30b63c1cd7ac0882408adb56
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679799 sha256=ca467785e80e0cdb4e7ea0dc8013cd43e0d7e6dd196e6a6bfacf909456792b42
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50

In [2]:
import numpy as np
import pandas as pd
import time
from sklearn.preprocessing import StandardScaler
from annoy import AnnoyIndex
import faiss
import hnswlib

# ===============================
# 1. Load & Siapkan Dataset Sleep Health (5D)
# ===============================
df = pd.read_csv('Sleep_health_and_lifestyle_dataset.csv')
fitur_5d = ['Age', 'Sleep Duration', 'Quality of Sleep', 'Stress Level', 'Heart Rate']
X_base = StandardScaler().fit_transform(df[fitur_5d]).astype(np.float32)

# Simulasikan 100.000 data dari dataset Sleep Health dengan sedikit variasi acak
np.random.seed(42)
n_data = 100_000
dim = 5
repeats = int(np.ceil(n_data / len(X_base)))
X = np.tile(X_base, (repeats, 1))[:n_data].copy()
X += np.random.normal(0, 0.01, X.shape).astype(np.float32)

# Query point (ambil profil baris ke-0)
query = X[0:1].copy()
k = 10

# ===============================
# 2. Annoy
# ===============================
print("=== Annoy ===")
ann_index = AnnoyIndex(dim, 'euclidean')

start = time.time()
for i in range(n_data):
    ann_index.add_item(i, X[i])
ann_index.build(10)  # 10 trees
build_time = time.time() - start

start = time.time()
neighbors = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", neighbors[0][:5], "...")

# ===============================
# 3. FAISS (Flat Index)
# ===============================
print("\n=== FAISS (IndexFlatL2) ===")
faiss_index = faiss.IndexFlatL2(dim)

start = time.time()
faiss_index.add(X)
build_time = time.time() - start

start = time.time()
distances, indices = faiss_index.search(query, k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", indices[0][:5], "...")

# ===============================
# 4. HNSW (hnswlib)
# ===============================
print("\n=== HNSW (hnswlib) ===")
hnsw_index = hnswlib.Index(space='l2', dim=dim)

start = time.time()
hnsw_index.init_index(max_elements=n_data, ef_construction=100, M=16)
hnsw_index.add_items(X, num_threads=4)
build_time = time.time() - start

hnsw_index.set_ef(50)

start = time.time()
labels, distances = hnsw_index.knn_query(query, k=k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", labels[0][:5], "...")

=== Annoy ===
Build time: 3.5080089569091797 detik
Query time: 0.0003044605255126953 detik
Neighbors: [0, 33286, 25058, 38896, 61336] ...

=== FAISS (IndexFlatL2) ===
Build time: 0.0009667873382568359 detik
Query time: 0.0058345794677734375 detik
Neighbors: [    0 33286 25058 38896 57596] ...

=== HNSW (hnswlib) ===
Build time: 7.536219835281372 detik
Query time: 0.00016236305236816406 detik
Neighbors: [    0 33286 25058 38896 61336] ...


Lakukan percobaan pada metric distance inner product. catat hasilnya pada tabel yang anda buat sendiri seperti pada praktikum 1.

In [3]:
import numpy as np
import time
import pandas as pd
from sklearn.preprocessing import StandardScaler
from annoy import AnnoyIndex
import faiss
import hnswlib

# ===============================
# 1. Konfigurasi Dataset Sleep Health
# ===============================
np.random.seed(42)
df = pd.read_csv('Sleep_health_and_lifestyle_dataset.csv')
fitur_5d = ['Age', 'Sleep Duration', 'Quality of Sleep', 'Stress Level', 'Heart Rate']
X_base = StandardScaler().fit_transform(df[fitur_5d]).astype(np.float32)

n_data = 100_000
dim = 5
k = 10

repeats = int(np.ceil(n_data / len(X_base)))
X = np.tile(X_base, (repeats, 1))[:n_data].copy()
X += np.random.normal(0, 0.01, X.shape).astype(np.float32)

# Normalisasi L2 agar Inner Product setara dengan Cosine Similarity
faiss.normalize_L2(X)

# Buat query dari baris ke-0
query = X[0:1].copy()
faiss.normalize_L2(query)

hasil_eksperimen = []

# ===============================
# 2. Annoy (Angular)
# ===============================
ann_index = AnnoyIndex(dim, 'angular')

start = time.time()
for i in range(n_data):
    ann_index.add_item(i, X[i])
n_trees = 10
ann_index.build(n_trees)
build_time = time.time() - start

start = time.time()
ann_neighbors = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
query_time = time.time() - start

ann_idx = ann_neighbors[0][:5]
str_idx = "[" + " ".join(map(str, ann_idx)) + "]"

hasil_eksperimen.append({
    'Metode': 'Annoy',
    'Distance Metrics': 'Angular',
    'Parameter': f'{n_trees} Trees',
    'Jumlah data': n_data,
    'Dimensi': dim,
    'Hasil Index terdekat (Top 5)': str_idx,
    'Waktu Build (s)': f'{build_time:.4f}',
    'Waktu Query (s)': f'{query_time:.4f}'
})

# ===============================
# 3. FAISS (IndexFlatIP)
# ===============================
faiss_index = faiss.IndexFlatIP(dim)

start = time.time()
faiss_index.add(X)
build_time = time.time() - start

start = time.time()
distances, indices = faiss_index.search(query, k)
query_time = time.time() - start

faiss_idx = indices[0][:5]
str_idx = "[" + " ".join(map(str, faiss_idx)) + "]"

hasil_eksperimen.append({
    'Metode': 'FAISS',
    'Distance Metrics': 'Inner Product',
    'Parameter': 'IndexFlatIP',
    'Jumlah data': n_data,
    'Dimensi': dim,
    'Hasil Index terdekat (Top 5)': str_idx,
    'Waktu Build (s)': f'{build_time:.4f}',
    'Waktu Query (s)': f'{query_time:.4f}'
})

# ===============================
# 4. HNSW (hnswlib)
# ===============================
hnsw_index = hnswlib.Index(space='ip', dim=dim)

start = time.time()
hnsw_index.init_index(max_elements=n_data, ef_construction=100, M=16)
hnsw_index.add_items(X, num_threads=4)
build_time = time.time() - start

ef_search = 50
hnsw_index.set_ef(ef_search)

start = time.time()
labels, distances = hnsw_index.knn_query(query, k=k)
query_time = time.time() - start

hnsw_idx = labels[0][:5]
str_idx = "[" + " ".join(map(str, hnsw_idx)) + "]"

hasil_eksperimen.append({
    'Metode': 'HNSW',
    'Distance Metrics': 'Inner Product',
    'Parameter': f'M=16, ef_construction=100, ef={ef_search}',
    'Jumlah data': n_data,
    'Dimensi': dim,
    'Hasil Index terdekat (Top 5)': str_idx,
    'Waktu Build (s)': f'{build_time:.4f}',
    'Waktu Query (s)': f'{query_time:.4f}'
})

# ===============================
# 5. Tampilkan tabel hasil
# ===============================
df_hasil = pd.DataFrame(hasil_eksperimen)
display(df_hasil)

,Metode,Distance Metrics,Parameter,Jumlah data,Dimensi,Hasil Index terdekat (Top 5),Waktu Build (s),Waktu Query (s)
0,Annoy,Angular,10 Trees,100000,5,[0 33286 68442 65450 61336],2.1647,0.0002
1,FAISS,Inner Product,IndexFlatIP,100000,5,[0 33286 68442 65450 61336],0.0009,0.0006
2,HNSW,Inner Product,"M=16, ef_construction=100, ef=50",100000,5,[0 33286 68442 65450 61336],5.6852,0.0002
